In [3]:
!pip install python-docx

In [1]:
"""
SE Weekly Report Processor – One‑Click GUI

What this does
- Pick an input folder of messy brand reports (.xlsx)
- Processes each file → cleans, pivots, and extracts metrics
- Saves per‑brand Cleaned Excel + Word Summary
- Builds a management summary workbook for all brands
- Simple GUI with 3 buttons: Select Input, Select Output, Process

Requirements
pip install pandas openpyxl python-docx tk
(Windows ships with tkinter; on Mac/Linux you may need to install it.)

To package as an .exe (optional):
pip install pyinstaller
pyinstaller --noconfirm --onefile --windowed se_report_gui.py
"""

import os
import sys
import traceback
from pathlib import Path
from datetime import datetime

import pandas as pd
from docx import Document

# --- GUI ---
import tkinter as tk
from tkinter import filedialog, messagebox

APP_TITLE = "SE Weekly Report Processor"


# -----------------------
# Core processing logic
# -----------------------

def process_brand_report(file_path: Path, output_folder: Path) -> dict:
    """Process a single brand report Excel file and return KPI metrics.

    Expected columns in sheet 0:
      - 'Activity Date'
      - 'Retailer Name'
      - 'Retailer City'
      - 'Label' (question/field)
      - 'Answer' (response/value)
    """
    brand_name = Path(file_path).stem.split()[0].upper()

    # Load messy report
    df = pd.read_excel(file_path, sheet_name=0, engine="openpyxl")
    df.columns = df.columns.astype(str).str.strip()

    # Validate minimal columns
    required_cols = {"Activity Date", "Retailer Name", "Retailer City", "Label", "Answer"}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {', '.join(missing)}")

    # Sort for consistency
    df = df.sort_values(by=["Retailer Name", "Label"], key=lambda x: x.astype(str).str.lower())

    # Pivot to wide format (first value if duplicates)
    pivot_df = (
        df.pivot_table(
            index=["Activity Date", "Retailer Name", "Retailer City"],
            columns="Label",
            values="Answer",
            aggfunc="first",
        )
        .reset_index()
    )

    # Normalize column names for robust matching
    pivot_df.columns = [str(c).strip() for c in pivot_df.columns]

    # Save cleaned Excel
    cleaned_excel_path = output_folder / f"Cleaned_{brand_name}_Report.xlsx"
    pivot_df.to_excel(cleaned_excel_path, index=False)

    # Helper counters
    def col_exists(name: str) -> bool:
        return name in pivot_df.columns

    def count_yes(col: str) -> int:
        if not col_exists(col):
            return 0
        return (
            pivot_df[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(["yes", "y", "true", "1"])
            .sum()
        )

    def count_not_none(col: str) -> int:
        if not col_exists(col):
            return 0
        s = pivot_df[col].astype(str).str.strip().str.lower()
        return s.ne("none").sum() - s.eq("").sum() - s.eq("nan").sum()

    def count_not_notlisted_like() -> int:
        # Count updates for any column containing shelf/facing etc.
        total = 0
        for c in pivot_df.columns:
            cl = c.lower()
            if ("shelf" in cl) or ("facing" in cl) or ("display" in cl):
                s = pivot_df[c].astype(str).str.strip().str.lower()
                total += s.ne("not listed").sum()
        return total

    # KPIs
    stores_covered = pivot_df["Retailer Name"].nunique()
    orders_generated = count_yes(f"Order Generated for {brand_name}?")
    restocks = orders_generated
    display_updates = count_not_notlisted_like()
    issues_found = count_not_none(f"Opportunities or Concerns for {brand_name}?")
    expiry_cases = count_yes(f"{brand_name} Products Expiring Within the Next 60 Days?")
    no_orders = max(stores_covered - orders_generated, 0)

    # Week ending = max Activity Date
    try:
        week_ending = pd.to_datetime(pivot_df["Activity Date"]).max().date()
    except Exception:
        week_ending = None

    # Create Word Summary
    doc = Document()
    doc.add_heading(f"Weekly Activity Summary – {brand_name}", level=1)
    if week_ending:
        doc.add_paragraph(f"Week Ending: {week_ending:%d %b %Y}\n")
    else:
        doc.add_paragraph("Week Ending: [Insert Date]\n")

    doc.add_paragraph(
        f"1. Stores Covered\n- Visited {stores_covered} modern retail outlets across key locations.\n"
    )
    doc.add_paragraph(
        "2. Sales Performance\n"
        f"- {orders_generated} confirmed orders generated.\n"
        f"- {no_orders} store visits with no orders placed this week.\n"
    )
    doc.add_paragraph(
        f"3. Restocks\n- {restocks} restocking activities completed to replenish shelves.\n"
    )
    doc.add_paragraph(
        f"4. Display & Merchandising\n- Shelf displays refreshed or improved in {display_updates} stores.\n"
    )
    doc.add_paragraph(
        "5. Issues & Resolutions\n"
        f"- {issues_found} issues/concerns identified, mainly relating to stock availability, display gaps, or competitor activity.\n"
        f"- {expiry_cases} cases of products expiring within 60 days were flagged and escalated for prompt action.\n"
        "Resolved through targeted restocking, shelf adjustments, and expiry stock management.\n"
    )

    summary_word_path = output_folder / f"Summary_{brand_name}.docx"
    doc.save(summary_word_path)

    return {
        "Brand": brand_name,
        "Stores Covered": stores_covered,
        "Orders Generated": orders_generated,
        "Restocks": restocks,
        "Display Updates": display_updates,
        "Issues Found": issues_found,
        "Expiry Cases": expiry_cases,
        "No Orders": no_orders,
        "Cleaned File": str(cleaned_excel_path),
        "Summary Doc": str(summary_word_path),
    }


def process_all_reports(input_folder: Path, base_output_folder: Path) -> Path:
    input_folder = Path(input_folder)
    base_output_folder = Path(base_output_folder)

    if not input_folder.exists() or not any(input_folder.glob("*.xlsx")):
        raise FileNotFoundError("No .xlsx files found in the selected input folder.")

    # Create dated output folder
    date_str = datetime.now().strftime("%Y-%m-%d")
    output_folder = base_output_folder / f"weekly_reports_output_{date_str}"
    output_folder.mkdir(parents=True, exist_ok=True)

    all_metrics = []
    errors = []

    for file in input_folder.glob("*.xlsx"):
        try:
            metrics = process_brand_report(file, output_folder)
            all_metrics.append(metrics)
        except Exception as e:
            errors.append((file.name, str(e)))

    # Save summary
    if all_metrics:
        summary_df = pd.DataFrame(all_metrics)
        summary_excel_path = output_folder / "Weekly_Brand_Summary.xlsx"
        summary_df.to_excel(summary_excel_path, index=False)

    # Save error log (if any)
    if errors:
        err_df = pd.DataFrame(errors, columns=["File", "Error"])
        err_path = output_folder / "_errors.csv"
        err_df.to_csv(err_path, index=False)

    return output_folder


# -----------------------
# Tkinter GUI
# -----------------------
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title(APP_TITLE)
        self.geometry("640x300")
        self.resizable(False, False)

        self.input_folder: Path | None = None
        self.output_folder: Path | None = None

        # Widgets
        self.lbl_in = tk.Label(self, text="Input Folder: (where the .xlsx files are)")
        self.ent_in = tk.Entry(self, width=70)
        self.btn_in = tk.Button(self, text="Select Input Folder", command=self.choose_input)

        self.lbl_out = tk.Label(self, text="Output Folder: (where results will be saved)")
        self.ent_out = tk.Entry(self, width=70)
        self.btn_out = tk.Button(self, text="Select Output Folder", command=self.choose_output)

        self.btn_run = tk.Button(self, text="Process Reports", command=self.run_processing, height=2)
        self.txt_log = tk.Text(self, height=8, width=80, state="disabled")

        # Layout
        pady = 6
        self.lbl_in.pack(anchor="w", padx=12, pady=(12, 0))
        self.ent_in.pack(fill="x", padx=12)
        self.btn_in.pack(anchor="e", padx=12, pady=(0, pady))

        self.lbl_out.pack(anchor="w", padx=12)
        self.ent_out.pack(fill="x", padx=12)
        self.btn_out.pack(anchor="e", padx=12, pady=(0, pady))

        self.btn_run.pack(pady=(8, 6))
        self.txt_log.pack(padx=12, pady=(0, 12))

    def choose_input(self):
        folder = filedialog.askdirectory(title="Select Input Folder")
        if folder:
            self.input_folder = Path(folder)
            self.ent_in.delete(0, tk.END)
            self.ent_in.insert(0, str(self.input_folder))

    def choose_output(self):
        folder = filedialog.askdirectory(title="Select Output Folder")
        if folder:
            self.output_folder = Path(folder)
            self.ent_out.delete(0, tk.END)
            self.ent_out.insert(0, str(self.output_folder))

    def log(self, message: str):
        self.txt_log.configure(state="normal")
        self.txt_log.insert("end", message + "\n")
        self.txt_log.see("end")
        self.txt_log.configure(state="disabled")
        self.update_idletasks()

    def run_processing(self):
        try:
            if not self.input_folder:
                raise ValueError("Please select an input folder.")
            if not self.output_folder:
                raise ValueError("Please select an output folder.")

            self.log("Starting processing… this may take a moment…")
            out_dir = process_all_reports(self.input_folder, self.output_folder)
            self.log(f"Done. Output saved to: {out_dir}")
            messagebox.showinfo(APP_TITLE, f"Processing completed.\nOutput: {out_dir}")
        except Exception as e:
            err = f"Error: {e}\n{traceback.format_exc()}"
            self.log(err)
            messagebox.showerror(APP_TITLE, str(e))


def main():
    app = App()
    app.mainloop()


if __name__ == "__main__":
    main()
